In [1]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
experimento = mlflow.set_experiment(
    "Sistema de alerta temprana de volatilidad para acciones"
)

print("URI activa:", mlflow.get_tracking_uri())
print("Experimento:", experimento.name)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


URI activa: http://127.0.0.1:5000
Experimento: Sistema de alerta temprana de volatilidad para acciones


In [2]:
import os
import sys
from hashlib import sha256
from pathlib import Path

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

from src.datos import cargar_obt

df = cargar_obt()

snapshot = df.sort_values(["TRADE_DATE", "SYMBOL"])
snapshot_bytes = snapshot.to_csv(
    index=False,
    lineterminator="\n",
).encode("utf-8")

datos_corrida = {
    "tabla_origen": "PSET2_DB.OBT.AAPL_DAILY_OBT",
    "fecha_corte_snapshot": os.environ["DATA_CUTOFF"],
    "sha256_snapshot_obt": sha256(snapshot_bytes).hexdigest(),
    "filas_snapshot": len(snapshot),
}

print("Dimensiones de la OBT:", df.shape)
print(datos_corrida)

Dimensiones de la OBT: (4087, 19)
{'tabla_origen': 'PSET2_DB.OBT.AAPL_DAILY_OBT', 'fecha_corte_snapshot': '2026-09-30', 'sha256_snapshot_obt': '9e9d8a678f5feb51385ae330e460a538835644c6e2068e9f989fd362ce45d4e8', 'filas_snapshot': 4087}


In [3]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

from src.features import construir_features

horizonte = 5
inicio_prueba = pd.Timestamp("2023-07-11")

features = construir_features(df)

vol_futura_5 = (
    features["RET_LAG_0"]
    .rolling(horizonte, min_periods=horizonte)
    .std(ddof=1)
    .shift(-horizonte)
    .rename("VOL_FUTURA_5")
)

# Calcular el fin de cada ventana antes de excluir las ruedas iniciales.
fin_ventana_target = features.index.to_series().shift(-horizonte)

datos_modelado = (
    features.iloc[60:]
    .join(vol_futura_5)
    .dropna(subset=["VOL_FUTURA_5"])
)

fin_ventana_modelado = fin_ventana_target.reindex(datos_modelado.index)
entrenamiento = datos_modelado.loc[
    fin_ventana_modelado < inicio_prueba
].copy()

X_desarrollo = entrenamiento.drop(columns="VOL_FUTURA_5")
vol_desarrollo = entrenamiento["VOL_FUTURA_5"]
columnas_numericas = X_desarrollo.columns.drop("MES").tolist()

cv_temporal = TimeSeriesSplit(n_splits=5, gap=horizonte)

print("Dataset supervisado:", datos_modelado.shape)
print("Features de entrenamiento:", X_desarrollo.shape)
print("Última fecha de alerta en entrenamiento:", X_desarrollo.index.max().date())
print("Inicio del hold-out:", inicio_prueba.date())

Dataset supervisado: (4022, 19)
Features de entrenamiento: (3212, 18)
Última fecha de alerta en entrenamiento: 2023-06-30
Inicio del hold-out: 2023-07-11


## Dummy Classifier

In [ ]:
#BASELINE DEL DUMMY CLASSIFIER

from time import perf_counter

from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

from src.pipeline import crear_pipeline

resultados_dummy = []

for pliegue, (idx_train, idx_val) in enumerate(
    cv_temporal.split(X_desarrollo), start=1
):
    X_train = X_desarrollo.iloc[idx_train]
    X_val = X_desarrollo.iloc[idx_val]

    vol_train = vol_desarrollo.iloc[idx_train]
    vol_val = vol_desarrollo.iloc[idx_val]

    p80 = vol_train.quantile(0.80)
    y_train = vol_train.gt(p80).astype(int)
    y_val = vol_val.gt(p80).astype(int)

    dummy = crear_pipeline(
        DummyClassifier(strategy="most_frequent"),
        columnas_numericas,
    )

    inicio = perf_counter()
    dummy.fit(X_train, y_train)
    segundos_ajuste = perf_counter() - inicio

    proba_train = dummy.predict_proba(X_train)[:, 1]
    proba_val = dummy.predict_proba(X_val)[:, 1]

    resultados_dummy.append({
        "pliegue": pliegue,
        "p80_train": p80,
        "positivos_validacion": int(y_val.sum()),
        "roc_auc_train": roc_auc_score(y_train, proba_train),
        "roc_auc_val": roc_auc_score(y_val, proba_val),
        "segundos_ajuste": segundos_ajuste,
    })

resultados_dummy = pd.DataFrame(resultados_dummy)
display(resultados_dummy.round(4))

,pliegue,p80_train,positivos_validacion,roc_auc_train,roc_auc_val,segundos_ajuste
0,1,1.8833,106,0.5,0.5,0.0118
1,2,1.8898,110,0.5,0.5,0.0510
2,3,1.8949,105,0.5,0.5,0.0300
3,4,1.8921,218,0.5,0.5,0.0403
4,5,2.0642,133,0.5,0.5,0.0246


In [5]:
import subprocess
from time import perf_counter

import mlflow.sklearn
from mlflow.models import infer_signature

commit_git = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
).strip()

cambios_sin_commit = bool(subprocess.check_output(
    ["git", "status", "--porcelain", "-uno"],
    cwd=raiz,
    text=True,
).strip())

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

p80_desarrollo = vol_desarrollo.quantile(0.80)
y_desarrollo = vol_desarrollo.gt(p80_desarrollo).astype(int)

dummy_reajustado = crear_pipeline(
    DummyClassifier(strategy="most_frequent"),
    columnas_numericas,
)

inicio = perf_counter()
dummy_reajustado.fit(X_desarrollo, y_desarrollo)
segundos_reajuste = perf_counter() - inicio

ejemplo = X_desarrollo.head(5)
firma = infer_signature(ejemplo, dummy_reajustado.predict_proba(ejemplo))

auc_train = resultados_dummy["roc_auc_train"].mean()
auc_val = resultados_dummy["roc_auc_val"].mean()

with mlflow.start_run(run_name="baseline_dummy_most_frequent") as corrida:
    mlflow.set_tags({
        "fase": "baseline",
        "familia": "DummyClassifier",
        "git_commit": commit_git,
        "git_cambios_sin_commit": str(cambios_sin_commit).lower(),
    })

    mlflow.log_params({
        **datos_corrida,
        "sha256_dataset_desarrollo": hash_desarrollo,
        "estrategia": "most_frequent",
        "semilla": "no_aplica",
        "n_pliegues": cv_temporal.n_splits,
        "gap": horizonte,
        "inicio_holdout": inicio_prueba.date().isoformat(),
        "p80_desarrollo": float(p80_desarrollo),
    })

    mlflow.log_metrics({
        "roc_auc_train_media_cv": float(auc_train),
        "roc_auc_val_media_cv": float(auc_val),
        "roc_auc_val_std_cv": float(resultados_dummy["roc_auc_val"].std(ddof=1)),
        "brecha_roc_auc_cv": float(auc_train - auc_val),
        "segundos_ajuste_cv_total": float(resultados_dummy["segundos_ajuste"].sum()),
        "segundos_reajuste_desarrollo": segundos_reajuste,
    })

    mlflow.log_text(
        resultados_dummy.to_csv(index=False),
        "resultados_por_pliegue.csv",
    )

    modelo_registrado = mlflow.sklearn.log_model(
        dummy_reajustado,
        name="pipeline_dummy",
        signature=firma,
        input_example=ejemplo,
        pyfunc_predict_fn="predict_proba",
    )

print("Run ID:", corrida.info.run_id)
print("URI del modelo:", modelo_registrado.model_uri)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/22 15:38:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run baseline_dummy_most_frequent at: http://127.0.0.1:5000/#/experiments/1/runs/10368c341481406098438197c2a16f14
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 10368c341481406098438197c2a16f14
URI del modelo: models:/m-aa55df3fbb444158b5017418e9903a39


## Baseline Persistencia

In [6]:
#BASELINE "Random Walk" para volatilidad (persistencia)

resultados_persistencia = []

for pliegue, (idx_train, idx_val) in enumerate(
    cv_temporal.split(X_desarrollo), start=1
):
    vol_train = vol_desarrollo.iloc[idx_train]
    vol_val = vol_desarrollo.iloc[idx_val]

    p80 = vol_train.quantile(0.80)
    y_train = vol_train.gt(p80).astype(int)
    y_val = vol_val.gt(p80).astype(int)

    score_train = X_desarrollo.iloc[idx_train]["VOL_HIST_5"]
    score_val = X_desarrollo.iloc[idx_val]["VOL_HIST_5"]
    alertas_val = score_val.gt(p80)

    resultados_persistencia.append({
        "pliegue": pliegue,
        "p80_train": p80,
        "positivos_validacion": int(y_val.sum()),
        "alertas_validacion": int(alertas_val.sum()),
        "roc_auc_train": roc_auc_score(y_train, score_train),
        "roc_auc_val": roc_auc_score(y_val, score_val),
    })

resultados_persistencia = pd.DataFrame(resultados_persistencia)
display(resultados_persistencia.round(4))

auc_cv = resultados_persistencia["roc_auc_val"]
print(f"ROC-AUC de validación: {auc_cv.mean():.4f} ± {auc_cv.std(ddof=1):.4f}")

,pliegue,p80_train,positivos_validacion,alertas_validacion,roc_auc_train,roc_auc_val
0,1,1.8833,106,109,0.5308,0.5771
1,2,1.8898,110,110,0.5613,0.5978
2,3,1.8949,105,105,0.5702,0.7414
3,4,1.8921,218,216,0.6170,0.7113
4,5,2.0642,133,133,0.6707,0.6989


ROC-AUC de validación: 0.6653 ± 0.0731


In [ ]:
from hashlib import sha256
import subprocess

commit_git = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
).strip()

cambios_sin_commit = bool(subprocess.check_output(
    ["git", "status", "--porcelain", "-uno"], cwd=raiz, text=True
).strip())

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

auc_train = resultados_persistencia["roc_auc_train"].mean()
auc_val = resultados_persistencia["roc_auc_val"].mean()
p80_desarrollo = float(vol_desarrollo.quantile(0.80))

with mlflow.start_run(run_name="baseline_persistencia_vol5") as corrida:
    mlflow.set_tags({
        "fase": "baseline",
        "familia": "persistencia",
        "git_commit": commit_git,
        "git_cambios_sin_commit": str(cambios_sin_commit).lower(),
    })

    mlflow.log_params({
        **datos_corrida,
        "sha256_dataset_desarrollo": hash_desarrollo,
        "estrategia": "volatilidad_historica_como_pronostico",
        "feature_score": "VOL_HIST_5",
        "semilla": "no_aplica",
        "n_pliegues": cv_temporal.n_splits,
        "gap": horizonte,
        "inicio_holdout": inicio_prueba.date().isoformat(),
        "p80_desarrollo": p80_desarrollo,
    })

    mlflow.log_metrics({
        "roc_auc_train_media_cv": float(auc_train),
        "roc_auc_val_media_cv": float(auc_val),
        "roc_auc_val_std_cv": float(
            resultados_persistencia["roc_auc_val"].std(ddof=1)
        ),
        "brecha_roc_auc_cv": float(auc_train - auc_val),
    })

    mlflow.log_text(
        resultados_persistencia.to_csv(index=False),
        "resultados_por_pliegue.csv",
    )

    mlflow.log_dict({
        "score": "VOL_HIST_5",
        "columna_para_estimar_umbral": "VOL_FUTURA_5",
        "cuantil": 0.80,
        "condicion_alerta": "score > umbral",
        "p80_desarrollo": p80_desarrollo,
    }, "regla_persistencia.json")

print("Run ID:", corrida.info.run_id)